# unbroadcast-pattern — ex2: unbroadcast — combined leading + size-1 case AND idempotence

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `unbroadcast-pattern`. Running the final beacon cell reports progress against the `Backprop: Unbroadcast pattern` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Unbroadcast pattern` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`unbroadcast-pattern`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "unbroadcast-pattern"
DD_SUBTOPIC = "Backprop: Unbroadcast pattern"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Unbroadcast — quick refresher

`unbroadcast(grad, original)` peels leading axes then collapses expanded size-1 axes so `grad.shape == original.shape`.

**Worked exemplar.** `original.shape == (1, 4)`, `grad.shape == (2, 3, 1, 4)`:
1. Peel two leading axes: sum dim=0 twice → `(1, 4)`.
2. No size-1 collapse needed (shapes already match).
Result: shape `(1, 4)`, values = sum of 6 broadcast copies.

### Exercise 2 — unbroadcast — combined leading + size-1 case AND idempotence

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply unbroadcast to the combined case — broadcasting added BOTH leading axes AND expanded a size-1 dim — and verify the function is idempotent (unbroadcast(unbroadcast(g, x), x) == unbroadcast(g, x)).
> Keywords: unbroadcast, leading-axes, size-1-axes, idempotence
> ```

**KCs targeted:** `unbroadcast-pattern`, `unbroadcast-is-idempotent`

Implement `unbroadcast(grad: Tensor, original: Tensor) -> Tensor` returning a tensor with shape `original.shape` and values summed along the broadcast axes.

Two passes (order matters):
1. **Peel leading axes** while `grad.ndim > original.ndim`: `grad = grad.sum(dim=0)`.
2. **Collapse size-1 axes** that got expanded: for each axis `i` where `original.shape[i] == 1` and `grad.shape[i] != 1`, `grad = grad.sum(dim=i, keepdim=True)`.

**Why this drill's facet matters.** ex1 tested each pass in isolation — leading-axes-only and size-1-only. ex2 tests the COMBINED case: e.g. `original.shape == (1, 4)`, `grad.shape == (2, 3, 1, 4)`. Step 1 peels TWO leading axes (yielding `(1, 4)`); step 2 finds no size-1 expansion to undo. But if you reordered the passes, you'd try to sum a missing axis and crash — order locks in.

ex2 also verifies **idempotence**: once `grad.shape == original.shape`, a second call must be a no-op (returning a tensor with the same shape and values). The function is its own fixed-point under repeated application.

In [ ]:
def unbroadcast(grad, original):
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(original.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad


<details><summary>Solution</summary>

```python
def unbroadcast(grad, original):
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(original.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad
```

**Why the two passes in THIS order.** The leading-axes pass reduces `grad.ndim` until it matches `original.ndim`. Only after that match is reached can the size-1 pass index axes by position — `original.shape[i]` and `grad.shape[i]` must refer to the same axis. Reversing the order would index past the end (or hit the wrong axis) and either crash or silently produce a wrong shape.

**Why `keepdim=True` in pass 2 but NOT pass 1.** Pass 1 is DROPPING axes (going from `ndim=4` to `ndim=2`, say) — `keepdim` would defeat the purpose. Pass 2 must PRESERVE the size-1 axis because `original.shape` has size-1 at that position; removing it would produce a shape that doesn't match `original`.

**Idempotence as a sanity invariant.** If shapes already match, the while-loop body doesn't run AND the for-loop's `if size == 1 and grad.shape[i] != 1` is always False (because the shapes match). Result: `grad` is returned unchanged. Applying `unbroadcast` to a tensor that already matches its target is a no-op — which is exactly what "idempotent" means.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()